# AlphaEvolve + **Qwen3-VL-2B-Instruct** on **CharXiv** — Google Colab

Runs **AlphaEvolve** (`evolve_instruct.py`) on **CharXiv** (`evaluate.py`). The search mutates `seed_instruct.py`; weights stay **Qwen/Qwen3-VL-2B-Instruct**.

**Optional uploads:** After the project folder exists, you may upload your own `evolve_instruct.py` and `evolved_instruct.py` into `evolving-vlm-inference-main/` (Colab left sidebar) to replace the repo copies before running evolution or evaluation.

**Colab:** **Runtime → Change runtime type → GPU** (T4+). **Secrets:** `HF_TOKEN`, `GEMINI_API_KEY` (or use prompts / `huggingface_hub.login()`).

**Project path:** `/content/evolving-vlm-inference-main` — upload a zip of the repo there **or** set `GIT_REPO_URL` in the first code cell.


In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path(os.environ.get("COLAB_PROJECT_ROOT", "/content/evolving-vlm-inference-main"))
GIT_REPO_URL = os.environ.get("GIT_REPO_URL", "")

NUM_GENERATIONS = int(os.environ.get("NUM_GENERATIONS", "3"))
POPULATION_SIZE = int(os.environ.get("POPULATION_SIZE", "6"))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("NUM_GENERATIONS:", NUM_GENERATIONS, "POPULATION_SIZE:", POPULATION_SIZE)



PROJECT_ROOT: /content/evolving-vlm-inference-main
NUM_GENERATIONS: 3 POPULATION_SIZE: 6


In [2]:
%%capture
!pip install -q "transformers>=4.37.0" qwen-vl-utils torch torchvision pillow accelerate google-generativeai bitsandbytes datasets huggingface_hub



In [3]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU. In Colab: Runtime → Change runtime type → GPU (e.g. T4), then re-run."
    )
print("GPU:", torch.cuda.get_device_name(0))



GPU: NVIDIA L4


In [4]:
import zipfile
from pathlib import Path

zip_path = Path("/content/evolving-vlm-inference-main.zip")  # change if your file name differs
out_dir = Path("/content")

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(out_dir)

# If the zip already contains a top-level folder "evolving-vlm-inference-main", you're done.
# If it extracted into a wrong name, rename so this path exists:
assert (out_dir / "evolving-vlm-inference-main").is_dir(), list(out_dir.iterdir())

In [5]:
import os, shutil, subprocess
from pathlib import Path

if GIT_REPO_URL:
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)
    subprocess.run(["git", "clone", "--depth", "1", GIT_REPO_URL, str(PROJECT_ROOT)], check=True)
elif not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"{PROJECT_ROOT} not found. Upload evolving-vlm-inference-main to /content or set GIT_REPO_URL."
    )

os.chdir(PROJECT_ROOT)
print("cwd:", os.getcwd())



cwd: /content/evolving-vlm-inference-main


In [6]:
import subprocess
from pathlib import Path

img_dir = PROJECT_ROOT / "charxiv" / "images"
img_dir.mkdir(parents=True, exist_ok=True)
has_img = any(img_dir.glob("*.jpg")) or any(img_dir.glob("*.png"))
if not has_img:
    zip_path = img_dir / "images.zip"
    url = "https://huggingface.co/datasets/princeton-nlp/CharXiv/resolve/main/images.zip"
    subprocess.run(["wget", "-q", "-O", str(zip_path), url], check=True)
    subprocess.run(["unzip", "-q", "-o", str(zip_path), "-d", str(img_dir)], check=True)
    print("Images extracted to", img_dir)
else:
    print("CharXiv images already present under", img_dir)



CharXiv images already present under /content/evolving-vlm-inference-main/charxiv/images


In [7]:
# Optional: replace evolution driver / last evolved program (use Colab file browser or files.upload())
# from google.colab import files
# uploaded = files.upload()
# for name, data in uploaded.items():
#     if name in ("evolve_instruct.py", "evolved_instruct.py"):
#         (PROJECT_ROOT / name).write_bytes(data)
#         print("Saved", PROJECT_ROOT / name)
print("Optional: upload evolve_instruct.py / evolved_instruct.py into", PROJECT_ROOT)



Optional: upload evolve_instruct.py / evolved_instruct.py into /content/evolving-vlm-inference-main


In [8]:
import os
from google.colab import userdata

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    from huggingface_hub import login
    login()

try:
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    pass

if not os.environ.get("GEMINI_API_KEY"):
    from getpass import getpass
    os.environ["GEMINI_API_KEY"] = getpass("Enter GEMINI_API_KEY: ")

assert os.environ.get("GEMINI_API_KEY"), "GEMINI_API_KEY is required."



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Enter GEMINI_API_KEY: ··········


In [9]:
import os
import sys
import importlib.util
from pathlib import Path

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))


def load_module(name: str, path: Path):
    spec = importlib.util.spec_from_file_location(name, path)
    m = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(m)
    return m


from evaluate import load_charxiv_data

queries, desc_data, rsn_data = load_charxiv_data(1, 0, 0, 0)
sample_key = next(iter(queries))
q0 = queries[sample_key]
fid, sub = sample_key.rsplit("_", 1)
gold = desc_data[fid]["answers"][int(sub)]
SAMPLE = {
    "key": sample_key,
    "image_path": q0["figure_path"],
    "question": q0["question"],
    "qid": q0.get("qid"),
    "gold": gold,
}
print("Sample key:", SAMPLE["key"], "gold:", SAMPLE["gold"])



Sample key: 0_0 gold: 60


/content/evolving-vlm-inference-main/charxiv/src/constants.py:255: SyntaxWarning: invalid escape sequence '\g'
  Response 3: \alpha, \beta, \gamma^t_v
/content/evolving-vlm-inference-main/charxiv/src/constants.py:339: SyntaxWarning: invalid escape sequence '\l'
  * Question: What is the name of the curve that intersects y=\lambda exactly three times?


In [10]:
initial = load_module("alphaevolve_seed_colab", PROJECT_ROOT / "seed_instruct.py")
before_pred = initial.vlm_inference(SAMPLE["image_path"], SAMPLE["question"], SAMPLE["qid"])
print("=== EXAMPLE — generation 0 (seed_instruct.py) ===")
print("Prediction:", before_pred)
print("Gold      :", SAMPLE["gold"])


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== EXAMPLE — generation 0 (seed_instruct.py) ===
Prediction: The y-axis label is "Overall Acc. (%)" and the tick marks are labeled with values. The highest tick value on the y-axis is 60. This is the topmost tick on the y-axis in the subplot at row 1 and column 2.

The spatially highest labeled tick on the y-axis is 60.
Gold      : 60


In [11]:
from evaluate import evaluate

print("=== BENCHMARK — generation 0 ===")
bench_before = evaluate(initial, return_details=False)
for k in ("desc_accuracy", "rsn_accuracy", "num_errors", "total_time", "avg_time_per_query"):
    print(f"  {k}: {bench_before[k]}")


=== BENCHMARK — generation 0 ===
  desc_accuracy: 0.3828125
  rsn_accuracy: 0.046875
  num_errors: 0
  total_time: 499.7864396572113
  avg_time_per_query: 2.603054373214642


In [12]:
import gc, sys, torch
_mname = "alphaevolve_seed_colab"
if _mname in sys.modules:
    mod = sys.modules.pop(_mname)
    for attr in ("_model", "_processor"):
        if hasattr(mod, attr):
            setattr(mod, attr, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Released generation-0 module.")


Released generation-0 module.


In [13]:
import evolve_instruct

evolve_instruct.NUM_GENERATIONS = NUM_GENERATIONS
evolve_instruct.POPULATION_SIZE = POPULATION_SIZE
population, history = evolve_instruct.run_evolution()
print("History entries:", len(history))


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


AlphaEvolve — Qwen3-VL-2B-Instruct
Generations : 25  |  Population : 10
Gemini model: gemini-2.5-flash  (override with GEMINI_MODEL env; 2.0-flash is unavailable for new keys)
Train split : 102 desc + 51 rsn samples
Holdout     : 26 desc + 13 rsn samples
Fitness     : descriptive accuracy (exact-match, train split)
Saves best  : by holdout accuracy (anti-overfitting)
Crossover   : every 5 generations
Seeds       : seed_instruct.py + 0 extra variants  →  evolved_instruct.py

[Gen 0] Evaluating seed: seed_instruct.py


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

  Train acc  : 34.3%  |  Holdout acc : 53.8%  |  2.852s/q

  Best gen-0 seed by holdout: 53.8%  (train 34.3%)  → saved to evolved_instruct.py

[Gen 1/25]
  Parent: 34.3%/53.8% acc (gen 0)
  Evaluating child...


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

...

  Child : train=61.8%  holdout=61.5%  0.550s/q
  Best by holdout : 61.5%  (train 61.8%)  → saved to evolved_instruct.py

[Gen 24/25]
  Parent: 59.8%/53.8% acc (gen 13)
  Evaluating child...


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

  Child : train=57.8%  holdout=50.0%  0.575s/q
  Best by holdout : 61.5%  (train 61.8%)  → saved to evolved_instruct.py

[Gen 25/25]
  [CROSSOVER] A: 60.8%/53.8%  B: 58.8%/53.8%  (train/holdout)
  Evaluating child...


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

  Child : train=60.8%  holdout=53.8%  0.573s/q
  Best by holdout : 61.5%  (train 61.8%)  → saved to evolved_instruct.py

Evolution complete!
  Best by train   : 61.8%  holdout=61.5%
  Best by holdout : 61.5%  train=61.8%  ← saved to disk
  Best speed      : 61.8%  (0.550s/q)
  Best overall    : 61.8%  score=0.2808
  Log             : evolution_log_instruct.json
History entries: 26


In [14]:
evolved = load_module("evolved_instruct_colab", PROJECT_ROOT / "evolved_instruct.py")
after_pred = evolved.vlm_inference(SAMPLE["image_path"], SAMPLE["question"], SAMPLE["qid"])
print("=== EXAMPLE — after AlphaEvolve (evolved_instruct.py) ===")
print("Prediction:", after_pred)
print("Gold      :", SAMPLE["gold"])


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

=== EXAMPLE — after AlphaEvolve (evolved_instruct.py) ===
Prediction: 60
Gold      : 60


In [3]:
from evaluate import evaluate

print("=== BENCHMARK — after AlphaEvolve ===")
bench_after = evaluate(evolved, return_details=False)
for k in ("desc_accuracy", "rsn_accuracy", "num_errors", "total_time", "avg_time_per_query"):
    print(f"  {k}: {bench_after[k]}")
print("\n=== SUMMARY (CharXiv: seed → evolved) ===")
print(f"Descriptive: {bench_before['desc_accuracy']:.4f} → {bench_after['desc_accuracy']:.4f}")


ModuleNotFoundError: No module named 'evaluate'

In [2]:
from pathlib import Path
import gc
import sys
import torch
from IPython.display import display, Markdown, Image as IPyImage
from evaluate import find_contrastive_descriptive_sample


def _vram_detach(mod_name: str) -> None:
    if mod_name in sys.modules:
        m = sys.modules.pop(mod_name)
        for attr in ("_model", "_processor"):
            if hasattr(m, attr):
                setattr(m, attr, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# Drop ex1's evolved module so we can load seed for one full eval without two Qwen copies.
_vram_detach("evolved_instruct_colab")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

seed_v = load_module("_scan_seed", PROJECT_ROOT / "seed_instruct.py")
evo_v = load_module("_scan_evo", PROJECT_ROOT / "evolved_instruct.py")
pair = find_contrastive_descriptive_sample(seed_v, evo_v, max_failure_checks=120)
_vram_detach("_scan_seed")
_vram_detach("_scan_evo")

if pair is not None:
    display(Markdown("### Contrastive example (seed misses gold; evolved matches)"))
    p = Path(pair["image_path"])
    if not p.is_file():
        raise FileNotFoundError(p)
    display(IPyImage(filename=str(p)))
    q = pair["question"]
    display(Markdown("**Question (trim)**"))
    display(Markdown((q[:1500] + "…") if len(q) > 1500 else q))
    print("**Query key:**", pair["key"])
    print("**Gold:**", pair["gold"])
    display(Markdown("#### Seed (`seed_instruct.py`) — incorrect"))
    print(pair["seed_response"])
    display(Markdown("#### Evolved (`evolved_instruct.py`) — correct"))
    print(pair["evolved_response"])
else:
    display(Markdown("### No contrastive hit in scan — same fixed `SAMPLE` + cached EXAMPLE preds"))
    p = Path(SAMPLE["image_path"])
    display(IPyImage(filename=str(p)))
    q = SAMPLE["question"]
    display(Markdown("**Question (trim)**"))
    display(Markdown((q[:1500] + "…") if len(q) > 1500 else q))
    print("**Gold:**", SAMPLE["gold"])
    s_pred = globals().get("before_pred")
    e_pred = globals().get("after_pred")
    if s_pred is None or e_pred is None:
        raise RuntimeError("Re-run the two EXAMPLE cells above (need `before_pred` and `after_pred`).")
    display(Markdown("#### Seed"))
    print(s_pred)
    display(Markdown("#### Evolved"))
    print(e_pred)

# Restore `evolved` for the benchmark cell below
evolved = load_module("evolved_instruct_colab", PROJECT_ROOT / "evolved_instruct.py")
print("Reloaded `evolved` for the benchmark cell.")

ModuleNotFoundError: No module named 'evaluate'